# RAG & Ingestion Quality Report

This notebook measures the quality of an ingested textbook. **It is deliberately thin: the two
headline sections run the project's *own* code** — `scripts/verify_ingestion.py` for ingestion
coverage and `retrieval_eval.evaluate_golden()` for retrieval — so the numbers match what the
pipeline itself considers correct, not a parallel re-implementation. Four extra checks
(sentence boundaries, source linkage, equation validity, chunk coherence) cover things the project
does not yet measure. Every cell reports **summary numbers and charts, not raw document dumps.**

**Requirements**
- MongoDB running with the book ingested; a checkout of the `marking-tools` repo (for the project code).
- `pip install pymongo matplotlib pymupdf python-dotenv`. Optional: Node.js (exact KaTeX check).

**Sections**

| # | Section | Source of the numbers |
|---|---------|-----------------------|
| A | Ingestion coverage, embeddings, chapters, content | **project** — `verify_ingestion.verify()` |
| B | Retrieval hit@k / MRR / nDCG | **project** — `evaluate_golden()` |
| C | Chunk sentence-boundary quality | new (this notebook) |
| D | Source-paragraph linkage | new |
| E | Equation validity: junk, LaTeX-compile, true duplicates | new |
| F | Chunk coherence (LLM-judged, optional) | new, via project `llm_service` |


## 0 · Configuration

Point `REPO_ROOT` at the `marking-tools` checkout — the notebook imports the project's code from
there, so this is required, not optional. Leave `BOOK_ID` blank to auto-pick the book with the most
chunks. `PDF_PATH` is strongly recommended: page coverage is only trustworthy against a real PDF
(without one there is no ground truth for how many pages *should* exist, and the number self-inflates).
The two `RUN_*` flags gate the steps that call external LLM APIs.

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
REPO_ROOT   = ""          # path to the marking-tools checkout (required)
MONGODB_URL = "mongodb://localhost:27017/?directConnection=true"
DB_NAME     = "marking_tools"
BOOK_ID     = ""          # "" = auto-pick the book with the most text chunks
PDF_PATH    = ""          # optional but recommended: absolute path to the source PDF

RUN_RETRIEVAL_EVAL = True    # section B (needs an embedding API key in REPO_ROOT/.env)
GOLDEN_PATH        = ""      # hand-labelled golden set JSON; "" = auto-build from chunks (upper bound)
AUTO_GOLDEN_N      = 25
RUN_LLM_JUDGE      = False   # section F (needs an LLM key; ~25 small calls via the project's llm_service)
JUDGE_SAMPLE       = 20
# ──────────────────────────────────────────────────────────────────────────────
import os, re, sys, json, math, collections, threading, asyncio
from pathlib import Path

if not REPO_ROOT:
    for cand in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (cand / "backend" / "app").exists() and (cand / "scripts").exists():
            REPO_ROOT = str(cand); break
assert REPO_ROOT, "Set REPO_ROOT to the marking-tools checkout."
sys.path[:0] = [str(Path(REPO_ROOT) / "scripts"), str(Path(REPO_ROOT) / "backend")]
os.environ.setdefault("MONGODB_URL", MONGODB_URL)
try:
    from dotenv import load_dotenv; load_dotenv(Path(REPO_ROOT) / ".env")
except ImportError: pass
os.environ["MONGODB_URL"] = MONGODB_URL   # keep our target DB after .env load
print("REPO_ROOT:", REPO_ROOT)

RESULTS = []
def record(metric, measured, value, target, status):
    RESULTS.append(dict(metric=metric, measured=measured, value=value, target=target, status=status))
    print(f"[{status}] {metric}: {measured}  (target {target})")


## 1 · Connect and pick the book

Uses the project's own `verify_ingestion.connect()` so the connection matches the pipeline. Prints
every book found with its per-index document counts; override `BOOK_ID` above if the wrong one is picked.

In [ ]:
import verify_ingestion as vi
db = vi.connect(MONGODB_URL, DB_NAME)
COLLS = ["pdf_chunks", "math_index", "figure_index", "table_index", "book_exercises"]

counts = {}
for coll in COLLS:
    for bid in db[coll].distinct("book_id"):
        counts.setdefault(bid, {})[coll] = db[coll].count_documents({"book_id": bid})
print(f"{'book_id':<42}" + "".join(f"{c[:12]:>14}" for c in COLLS))
for bid, row in sorted(counts.items()):
    print(f"{bid:<42}" + "".join(f"{row.get(c, 0):>14}" for c in COLLS))

if not BOOK_ID:
    BOOK_ID = max(counts, key=lambda b: counts[b].get("pdf_chunks", 0))
B = {"book_id": BOOK_ID}
if not PDF_PATH:
    for cand in Path(REPO_ROOT).glob("Book/*.pdf"):
        if BOOK_ID.split("-")[0].lower() in cand.stem.lower(): PDF_PATH = str(cand); break
print(f"\nAuditing: {BOOK_ID}\nPDF: {PDF_PATH or '(none — page coverage will be reported as UNKNOWN)'}")


## 2 · Chart style

A colour-vision-safe palette with value labels on every bar (nothing depends on colour alone).
Green/amber/red are reserved for pass/warn/fail; blue is a plain magnitude.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
BLUE, GREEN, INK, MUTED, GRID = "#2a78d6", "#008300", "#0b0b0b", "#898781", "#e1e0d9"
GOOD, WARNC, CRIT = "#0ca30c", "#fab219", "#d03b3b"
SC = {"PASS": GOOD, "WARN": WARNC, "FAIL": CRIT, "INFO": BLUE, "SKIP": MUTED}

def style(ax, title):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.set_title(title, color=INK, fontsize=11, loc="left", pad=10, fontweight="bold")

def hbar(labels, values, title, fmt, colors=None, xmax=None):
    fig, ax = plt.subplots(figsize=(8, 0.55 + 0.45 * len(labels)))
    bars = ax.barh(labels, values, height=0.55, color=colors or BLUE)
    ax.bar_label(bars, labels=[fmt(v) for v in values], padding=4, color=INK, fontsize=9)
    ax.set_xlim(0, xmax or max(values) * 1.25 or 1); ax.invert_yaxis(); style(ax, title)
    plt.tight_layout(); plt.show()
print("style ready")


## A · Ingestion quality — via the project's `verify_ingestion.verify()`

**This is the project's own verification code, not a re-implementation.** One call returns page
coverage (with the pipeline's page-range clamp and its <20-char blank-page rule), embedding validity,
chapter structure, text quality, and math/table/image detection. We chart the results and record the
scorecard rows. Coverage is only scored PASS/FAIL when a real PDF gives a page count; without one it
is **UNKNOWN** (the number would otherwise self-normalise to ~100%). Embedding validity is a
**sanity check**, not a quality score: the ingestion writers drop any doc that fails to embed, so this
confirms nothing slipped through with a broken vector — it cannot measure docs lost at ingest time.

In [ ]:
r = vi.verify(BOOK_ID, PDF_PATH or None, db)

# Coverage
cov = r["coverage"]
if cov:
    status = "PASS" if not cov["truly_missing"] and cov["pct"] >= 98 else \
             ("WARN" if not cov["truly_missing"] else "FAIL")
    hbar(["indexed", "missing (blank)", "missing (content)"],
         [cov["covered"], len(cov["blank_pages"]), len(cov["truly_missing"])],
         f"Page coverage — {cov['pct']:.1f}% of {cov['expected']} pages", lambda v: f"{int(v)}",
         colors=[BLUE, MUTED, CRIT], xmax=cov["expected"] * 1.2)
    record("Content coverage", f"{cov['pct']:.1f}% ({cov['covered']}/{cov['expected']} pages)",
           cov["pct"], "≥98%", status)
    if cov["truly_missing"]:
        print(f"Substantive pages missing ({len(cov['truly_missing'])}):",
              cov["truly_missing"][:15], "…" if len(cov["truly_missing"]) > 15 else "")
else:
    record("Content coverage", "no PDF — cannot verify", None, "≥98%", "SKIP")

# Embedding sanity
e = r["embeddings"]
bad = e["missing"] + e["wrong_dim"]
record("Embedding validity (sanity)", f"{e['ok']}/{e['total']} chunks carry a valid 768-vector",
       100 if e["total"] and bad == 0 else 0, "no broken vectors", "PASS" if bad == 0 else "FAIL")

# Chapters + content richness
ch, c = r["chapters"], r["content"]
hbar(["math (font)", "math (LaTeX)", "tables", "images"],
     [c["has_math"], c["has_math_text"], c["has_tables"], c["has_images"]],
     f"Chunks carrying each content type (of {r['total_chunks']})", lambda v: f"{int(v)}")
print(f"Chapters detected: {ch['found'] or '— (single-chapter or unlabelled book)'}"
      + (f"  | missing: {ch['missing']}" if ch["missing"] else ""))
print(f"Avg chunk text: {r['text_quality']['avg_len']:.0f} chars  |  "
      f"issues from verify_ingestion: {r['issues'] or 'none'}")


## B · Retrieval quality — via the project's `evaluate_golden()`

**This routes real queries through the production retrieval path** (query embedding → specialist
indexes → fusion) and scores them with the project's own metric functions. With a hand-labelled
`GOLDEN_PATH` (e.g. `eval/golden/*.json`) the numbers are credible; without one the notebook
auto-builds queries from each chunk's own section title + key terms and expects that chunk's pages
back — an easier, self-referential test, so treat auto-mode as an **upper bound**. Reports hit@k
(found any relevant result), MRR (how high), and nDCG@k, per specialist index and fused.

In [ ]:
def _retrieval_eval():
    from app.services.retrieval_eval import evaluate_golden
    def run_async(coro):
        out = {}; t = threading.Thread(target=lambda: out.update(r=asyncio.run(coro))); t.start(); t.join()
        return out["r"]
    if GOLDEN_PATH:
        golden = json.loads(Path(GOLDEN_PATH).read_text()); mode = "hand-labelled golden set"
    else:
        mode = "auto (generic queries — indicative only; use a golden set for real numbers)"
        pool = list(db.pdf_chunks.find(B).sort("page_start", 1))
        pool = pool[::max(1, len(pool) // AUTO_GOLDEN_N)][:AUTO_GOLDEN_N]
        golden = {"book_id": BOOK_ID, "queries": [
            {"query": f"{c.get('section_title','')}: {', '.join((c.get('key_terms') or [])[:4])}",
             "expect": {"pages": list(range(int(c.get('page_start') or 1), int(c.get('page_end') or 1) + 1))}}
            for c in pool]}
    rep = run_async(evaluate_golden(golden, k=5))
    s = rep["summary"]; order = [n for n in ("text","formula","figure","table","fused") if n in s]
    fig, ax = plt.subplots(figsize=(9, 3)); x = range(len(order)); w = 0.27
    for i, (m, col) in enumerate([("hit@k", BLUE), ("mrr", GREEN), ("ndcg@k", MUTED)]):
        ax.bar([xi + (i - 1) * w for xi in x], [s[n][m] for n in order], width=w, label=m, color=col)
    ax.set_xticks(list(x)); ax.set_xticklabels(order); ax.set_ylim(0, 1.1); ax.legend(frameon=False, fontsize=9)
    style(ax, f"Retrieval — {rep['num_queries']} queries, {mode}"); plt.tight_layout(); plt.show()
    hit, mrr = s["fused"]["hit@k"], s["fused"]["mrr"]
    record(f"Retrieval hit@5 (fused, {mode.split()[0]})", f"{hit:.2f}", hit*100, "≥0.90", "PASS" if hit >= .9 else "WARN")
    record("Retrieval MRR (fused)", f"{mrr:.2f}", mrr*100, "report", "INFO")

if not RUN_RETRIEVAL_EVAL:
    print("Skipped (RUN_RETRIEVAL_EVAL = False)."); record("Retrieval hit@5 / MRR", "not run", None, "high", "SKIP")
else:
    try:
        _retrieval_eval()
    except Exception as ex:
        print("Retrieval eval failed (embedding key / provider?):", str(ex)[:140])
        record("Retrieval hit@5 / MRR", "eval failed — check embedding key", None, "high", "SKIP")


## C · Chunk sentence-boundary quality *(new)*

Checks whether each text chunk ends at a sentence boundary (`.`/`!`/`?`, optional closing bracket, or
`:`). A chunk cut mid-sentence feeds broken context to generation and confuses ranking. Target
**≥ 95%**. Only a small capped sample of offenders is shown — no full text dumps.

In [ ]:
SENT_END = re.compile(r"[.!?][)\]\"\u201d\u2019']*\s*$|:\s*$")
chunks = list(db.pdf_chunks.find(B, {"text": 1}))
ends = [(c, bool(SENT_END.search((c.get("text") or "").rstrip()))) for c in chunks]
good = sum(ok for _, ok in ends); rate = 100 * good / max(len(chunks), 1)
hbar(["ends at sentence", "cut mid-sentence"], [good, len(chunks) - good],
     f"Chunk endings — {rate:.1f}% clean of {len(chunks)}", lambda v: f"{int(v)}",
     colors=[GOOD, CRIT], xmax=len(chunks) * 1.2)
for c, ok in [ce for ce in ends if not ce[1]][:5]:
    print("  cut: …" + repr((c.get("text") or "").rstrip()[-60:]))
record("Chunk boundary quality", f"{rate:.1f}% ({good}/{len(chunks)})", rate, "≥95%",
       "PASS" if rate >= 95 else "FAIL")


## D · Source-paragraph linkage *(new)*

Every equation / figure / table document should carry a `parent_chunk_id` that resolves to a real
text chunk — this is what lets retrieval expand a formula hit into its explanation, and the basis for
"every answer traceable to the textbook". Skips any index the book doesn't use (no crash on a
figure-free book). Target **≥ 90%**.

In [ ]:
chunk_ids = {str(c["_id"]) for c in db.pdf_chunks.find(B, {"_id": 1})}
link = {}
for coll in ("math_index", "figure_index", "table_index"):
    docs = list(db[coll].find(B, {"parent_chunk_id": 1}))
    if docs:
        link[coll] = (sum(1 for d in docs if str(d.get("parent_chunk_id") or "") in chunk_ids), len(docs))
if not link:
    record("Source linkage", "no equation/figure/table docs", None, "≥90%", "SKIP")
else:
    hbar(list(link), [100 * ok / t for ok, t in link.values()], "Source-paragraph linkage",
         lambda v: f"{v:.1f}%", xmax=118)
    tot_ok = sum(ok for ok, _ in link.values()); tot = sum(t for _, t in link.values())
    pct = 100 * tot_ok / tot
    record("Source linkage", f"{pct:.1f}% ({tot_ok}/{tot})", pct, "≥90%", "PASS" if pct >= 90 else "FAIL")


## E · Equation index validity *(new)*

Three equation-index checks in one cell:
- **Junk docs** — vision-pass sentinels (`NO_MATH` / `NO_CHART`, the project's own sentinel strings)
  or sub-4-char fragments that slipped into the index. Target **~0%**.
- **True duplicates** — using the pipeline's *own* dedup key (`parent_chunk_id` + `normalise_formula`
  from `math_index`), how many documents are exact re-inserts the pipeline should have collapsed?
  Target **0**. (The same formula appearing under *different* parent chunks is reported separately as
  legitimate cross-location repeats, **not** counted as waste — the previous version wrongly did.)
- **LaTeX compile** — fraction of usable formulas that render. If Node + the project's KaTeX build are
  present it uses the exact frontend engine; otherwise it falls back to matplotlib mathtext (marked
  approximate). It also flags strings that are plain prose rather than math, since those "compile"
  without being real formulas.

In [ ]:
from app.services.math_index import normalise_formula
mdocs = list(db.math_index.find(B, {"formula_latex": 1, "page": 1, "parent_chunk_id": 1}))
def latex_of(d): return str(d.get("formula_latex") or "").strip()

# Junk (project sentinels + fragments)
def is_junk(d):
    t = latex_of(d)
    return ("NO_MATH" in t) or ("NO_CHART" in t) or len(t.replace("$", "").strip()) < 4
junk = sum(1 for d in mdocs if is_junk(d)); pj = 100 * junk / max(len(mdocs), 1)
record("Equation junk docs", f"{pj:.1f}% ({junk}/{len(mdocs)})", pj, "~0%",
       "PASS" if pj <= 5 else ("WARN" if pj <= 15 else "FAIL"))

# True duplicates (pipeline dedup key) vs legitimate cross-location repeats
usable = [d for d in mdocs if not is_junk(d)]
bykey = collections.Counter((str(d.get("parent_chunk_id") or ""), normalise_formula(latex_of(d))) for d in usable)
true_dups = sum(v - 1 for v in bykey.values() if v > 1)
byform = collections.Counter(normalise_formula(latex_of(d)) for d in usable)
cross = sum(v - 1 for v in byform.values() if v > 1) - true_dups
record("Equation true duplicates", f"{true_dups} (plus {cross} legit cross-location repeats)",
       true_dups, "0", "PASS" if true_dups == 0 else "FAIL")

# LaTeX compile
import subprocess, tempfile
texs = [latex_of(d).strip("$").strip() for d in usable]
katex = ""
kd = Path(REPO_ROOT) / "frontend" / "node_modules" / "katex"
if kd.exists(): katex = str(kd)
passed = failed = prose = 0; engine = "katex"
PROSE = re.compile(r"[A-Za-z]{4,}\s+[A-Za-z]{4,}")   # two long words in a row ⇒ likely a sentence
def looks_prose(t): return bool(PROSE.search(t)) and "\\" not in t and "_" not in t and "^" not in t
if katex:
    with tempfile.TemporaryDirectory() as td:
        Path(td, "f.json").write_text(json.dumps(texs))
        Path(td, "r.js").write_text(
            "const k=require(%r),fs=require('fs');const a=JSON.parse(fs.readFileSync(%r));"
            "const o=a.map(t=>{try{k.renderToString(t,{throwOnError:true,displayMode:true});return ''}"
            "catch(e){return String(e.message).slice(0,80)}});fs.writeFileSync(%r,JSON.stringify(o));"
            % (katex, str(Path(td, "f.json")), str(Path(td, "o.json"))))
        subprocess.run(["node", str(Path(td, "r.js"))], check=True, capture_output=True, timeout=300)
        outs = json.loads(Path(td, "o.json").read_text())
    for t, err in zip(texs, outs):
        if err: failed += 1
        else:
            passed += 1
            if looks_prose(t): prose += 1
else:
    engine = "mathtext (approx — install Node+katex for the exact check)"
    from matplotlib.mathtext import MathTextParser
    p = MathTextParser("agg")
    for t in texs:
        try: p.parse(f"${t}$"); passed += 1; prose += looks_prose(t)
        except Exception: failed += 1
rate = 100 * passed / max(passed + failed, 1)
hbar(["renders", "fails", "renders but is prose"], [passed, failed, prose],
     f"LaTeX validity — {rate:.1f}% render ({engine.split()[0]})", lambda v: f"{int(v)}",
     colors=[GOOD, CRIT, WARNC], xmax=(passed + failed) * 1.2)
record(f"LaTeX compile ({engine.split()[0]})", f"{rate:.1f}% render, {prose} are prose not math",
       rate, "100%", "PASS" if rate >= 99.95 and prose == 0 else ("WARN" if rate >= 90 else "FAIL"))


## F · Chunk coherence — LLM-judged *(optional, new)*

Only runs when `RUN_LLM_JUDGE = True`. Samples chunks and asks an LLM (through the project's own
`llm_service`, so it uses the configured provider fallback and key management — not a hand-rolled HTTP
call) to score each 0–1 for grammatical completeness, standalone coherence, and freedom from
extraction artifacts. Catches problems a regex cannot. **≥ 0.85 pass, < 0.70 fail.**

In [ ]:
if not RUN_LLM_JUDGE:
    print("Skipped (RUN_LLM_JUDGE = False)."); record("Chunk coherence (LLM)", "not run", None, "≥0.85", "SKIP")
else:
    from app.services.llm_service import llm_service
    def run_async(coro):
        out = {}; t = threading.Thread(target=lambda: out.update(r=asyncio.run(coro))); t.start(); t.join()
        return out["r"]
    sample = sorted(chunks, key=lambda c: str(c["_id"]))[::max(1, len(chunks)//JUDGE_SAMPLE)][:JUDGE_SAMPLE]
    P = ('Score this textbook chunk 0.0-1.0 for grammatical completeness, standalone coherence, and '
         'absence of OCR/extraction artifacts. Reply ONLY strict JSON {"score": <float>}.\n\nCHUNK:\n')
    async def judge_all():
        out = []
        for ch in sample:
            try:
                txt = await llm_service.generate(P + (ch.get("text") or "")[:5000])
                out.append(float(re.search(r'"score"\s*:\s*([0-9.]+)', txt).group(1)))
            except Exception as ex:
                print("  skip:", str(ex)[:60])
        return out
    scores = run_async(judge_all())
    if scores:
        mean = sum(scores) / len(scores)
        fig, ax = plt.subplots(figsize=(8, 2.6))
        ax.hist(scores, bins=[i/20 for i in range(21)], color=BLUE)
        for x, l in ((.70, "fail"), (.85, "pass")): ax.axvline(x, color=MUTED, ls="--", lw=1)
        style(ax, f"Chunk coherence — mean {mean:.2f} (n={len(scores)})"); plt.tight_layout(); plt.show()
        record("Chunk coherence (LLM)", f"{mean:.2f} mean, {sum(s<.7 for s in scores)}/{len(scores)} below 0.70",
               mean*100, "≥0.85", "PASS" if mean>=.85 else ("WARN" if mean>=.70 else "FAIL"))
    else:
        record("Chunk coherence (LLM)", "all calls failed (check keys)", None, "≥0.85", "SKIP")


## Scorecard

Every metric above, coloured by verdict. Percentage-style metrics share a 0–100 axis; each bar is
labelled with its measured value. A FAIL always has its evidence in the matching section.

In [ ]:
rows = [r for r in RESULTS if r["value"] is not None]
fig, ax = plt.subplots(figsize=(10, 0.6 + 0.45 * max(len(rows), 1)))
bars = ax.barh([r["metric"] for r in rows][::-1], [min(float(r["value"]), 100) for r in rows][::-1],
               height=0.55, color=[SC[r["status"]] for r in rows][::-1])
ax.bar_label(bars, labels=[f" {r['measured']}  [{r['status']}]" for r in rows[::-1]],
             padding=4, color=INK, fontsize=8.5)
ax.set_xlim(0, 170); ax.set_xticks([0, 25, 50, 75, 100]); style(ax, f"Quality scorecard — {BOOK_ID}")
plt.tight_layout(); plt.show()

w = max(len(r["metric"]) for r in RESULTS) + 2
print(f"{'metric':<{w}}{'measured':<44}{'target':<12}status")
for r in RESULTS:
    print(f"{r['metric']:<{w}}{r['measured']:<44}{r['target']:<12}{r['status']}")
print("\n" + " · ".join(f"{k} {sum(x['status']==k for x in RESULTS)}"
                         for k in ("PASS","WARN","FAIL","SKIP")))


## Reading the results

- **Sections A and B are the project itself** — if they pass, the pipeline's own verification passes.
  A **SKIP** on coverage means no PDF was supplied (supply one for a real number); a **SKIP** on
  retrieval means the flag is off or no embedding key is set.
- **Embedding validity is a sanity light, not a quality score** — it is designed to read 100% because
  the ingester drops un-embeddable docs; a FAIL here would mean genuine corruption.
- **Boundary / linkage / equation-validity (C–E)** are the checks the project does not yet run. A
  boundary FAIL points at the chunker; an equation-junk or duplicate FAIL points at the index writers.
- **For investor-grade retrieval numbers use a hand-labelled `GOLDEN_PATH`**, not auto-mode.

Re-run the whole notebook after any pipeline change — it is your content-quality regression dashboard.